# 05 — Prepare Delta table for **Mosaic AI Vector Search** (PDF chunks)

**Goal:** Build a small, stable **source Delta table** with a string **primary key** and the **text column** you want embedded, then create a **Delta Sync** index that uses a **Databricks Foundation Model** embedding endpoint (recommended: `databricks-qwen3-embedding-0-6b` for multilingual / Indic-friendly retrieval).

**Prerequisites:** notebooks **03** (`pdf_text_chunks`) and **04** completed.

**After this notebook:** In the Databricks UI (or SDK below), create a Vector Search **endpoint** (if you do not have one), then a **Delta Sync index** with:
- **Source table:** `ayurveda_lakehouse.ayurgenix.pdf_chunks_for_vector`
- **Primary key:** `chunk_pk`
- **Embedding source column:** `chunk_text`
- **Managed embeddings model:** `databricks-qwen3-embedding-0-6b` (or another FM embedding your workspace exposes)

Point the Streamlit app at the PDF index using `VECTOR_SEARCH_ENDPOINT` and `VECTOR_SEARCH_INDEX_PDF` (or legacy `VECTOR_SEARCH_INDEX`). For **AyurGenix without SQL warehouse**, run notebook **06** and set `VECTOR_SEARCH_INDEX_CURATED`.


In [ ]:
%sql
USE CATALOG ayurveda_lakehouse;
USE SCHEMA ayurgenix;


## Step 1 — Materialise a Vector Search–friendly table

Filters very short chunks (same idea as `vw_pdf_extraction_quality` in notebook 04). Adjust the `50` threshold if needed.


In [ ]:
%sql
CREATE OR REPLACE TABLE ayurveda_lakehouse.ayurgenix.pdf_chunks_for_vector AS
SELECT
  concat_ws(':', source_file, CAST(page_number AS STRING), CAST(chunk_index AS STRING)) AS chunk_pk,
  source_file,
  CAST(page_number AS INT) AS page_number,
  CAST(chunk_index AS INT) AS chunk_index,
  chunk_text
FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
WHERE length(trim(chunk_text)) >= 50;

SELECT COUNT(*) AS chunk_rows FROM ayurveda_lakehouse.ayurgenix.pdf_chunks_for_vector;


## Step 2 — (Optional) Create index via Python SDK

1. Replace `YOUR_VS_ENDPOINT_NAME` with an existing Vector Search endpoint name from **Compute → Vector Search**.
2. Replace index FQN if you prefer another catalog/schema.
3. Run once; if the index already exists, use the UI to edit or delete first.

Docs: [Query a vector search index](https://docs.databricks.com/en/vector-search/query-vector-search.html), [Foundation models](https://docs.databricks.com/en/machine-learning/foundation-model-apis/supported-models.html).


In [ ]:
%pip install -q databricks-vectorsearch
dbutils.library.restartPython()


In [ ]:
from databricks.vector_search.client import VectorSearchClient

VS_ENDPOINT = "YOUR_VS_ENDPOINT_NAME"  # e.g. "vs-endpoint-ayurveda"
INDEX_NAME = "ayurveda_lakehouse.ayurgenix.pdf_chunks_index"
SOURCE_TABLE = "ayurveda_lakehouse.ayurgenix.pdf_chunks_for_vector"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

vsc = VectorSearchClient(disable_notice=True)

# Uncomment to create (first time only):
# vsc.create_delta_sync_index(
#     endpoint_name=VS_ENDPOINT,
#     index_name=INDEX_NAME,
#     source_table_name=SOURCE_TABLE,
#     pipeline_type="TRIGGERED",
#     primary_key="chunk_pk",
#     embedding_source_column="chunk_text",
#     embedding_model_endpoint_name=EMBED_MODEL,
# )

idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
idx.sync()
print("Sync requested for", INDEX_NAME)


## Step 3 — Smoke test (SDK query)

If this returns rows, set the same `VS_ENDPOINT` and UC `INDEX_NAME` on your **Databricks App** as `VECTOR_SEARCH_ENDPOINT` and `VECTOR_SEARCH_INDEX`.


In [ ]:
from databricks.vector_search.client import VectorSearchClient

VS_ENDPOINT = "YOUR_VS_ENDPOINT_NAME"
INDEX_NAME = "ayurveda_lakehouse.ayurgenix.pdf_chunks_index"

vsc = VectorSearchClient(disable_notice=True)
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
rows = index.similarity_search(
    query_text="tulsi cough dosha",
    columns=["chunk_pk", "source_file", "page_number", "chunk_index", "chunk_text"],
    num_results=5,
    query_type="HYBRID",
)
print(rows)
